# 03. Duplicatas em compradores e vendedores

Criterio e justificativa por arquivo, e qual registro e mantido.

In [1]:
import pandas as pd
from pathlib import Path
from cristalux.config import settings
from cristalux.cleaning import parsers as p
from cristalux.cleaning.base import ler_csv
from cristalux.pipeline.run import ARQUIVOS, limpar_tudo, relatorio_qualidade

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200, "display.max_colwidth", 60)
raws, res = limpar_tudo(settings.data_dir)

## Compradores

**Chave de duplicidade: CNPJ normalizado (so digitos).** Razao social, cidade e e-mail variam de um cadastro para o outro
(`Auto Center Pinheiros Ltda` e `Auto Center Pinheiros`, `Sao Paulo` e `Sao Paulo`), entao nao servem de chave. O CNPJ identifica a empresa.

In [2]:
cr = raws["compradores"].copy()
cr["cnpj_digitos"] = cr["cnpj"].map(p.so_digitos)
grupos = cr[cr.duplicated("cnpj_digitos", keep=False) & cr["razao_social"].map(p.limpar).notna()].sort_values(["cnpj_digitos", "id_comprador"])
print(grupos["cnpj_digitos"].nunique(), "CNPJs com mais de um cadastro,", len(grupos), "linhas")
grupos[["id_comprador", "razao_social", "cnpj", "cidade", "estado", "email", "limite_credito"]].head(14)

13 CNPJs com mais de um cadastro, 26 linhas


,id_comprador,razao_social,cnpj,cidade,estado,email,limite_credito
4,C005,Distribuidora AutoPeças Sul,11.222.333/0001-44,Porto Alegre,RS,marcos@autopecassul.com.br,R$ 200.000
5,C006,AutoPeças Sul Distribuidora,11.222.333/0001-44,Porto Alegre,Rio Grande do Sul,marcos@autopecassul.com.br,200000.00
0,C001,Auto Center Pinheiros Ltda,12.345.678/0001-90,São Paulo,SP,roberto@acpinheiros.com.br,R$ 80.000
1,C002,Auto Center Pinheiros,12.345.678/0001-90,Sao Paulo,São Paulo,roberto@acpinheiros.com.br,80000
26,C027,Parceiro ABC,12.345.678/0001-99,São Paulo,SP,,R$ 10.000
27,C028,Parceiro ABC Ltda,12.345.678/0001-99,sp,SP,,10000
20,C021,Mega Auto Bahia,13.579.246/0001-80,Salvador,BA,luis@megaautoba.com.br,R$ 180.000
21,C022,Mega Auto Bahia Ltda,13.579.246/0001-80,Salvador,Bahia,luisaraujo@megaautoba.com.br,180000
16,C017,Centro Automotivo Goiás,22.333.444/0001-55,Goiânia,GO,renata@centroautogo.com,R$ 55.000
17,C018,Centro Automotivo GO,22.333.444/0001-55,goiania,Goias,rbraga@centroautogo.com,55000


**Registro mantido (RN-22):** o que tem mais campos preenchidos; no empate, o de menor id, que e o mais antigo e o que as vendas
provavelmente ja referenciam. Campos nulos do mantido sao preenchidos com os do descartado. Cada id descartado vira um alias do mantido,
e as vendas que apontavam para ele sao reatribuidas.

In [3]:
rc = res["compradores"]
print(rc.stats["linhas_brutas"], "->", rc.stats["linhas_limpas"], "compradores")
pd.DataFrame({"descartado": rc.stats["alias"].keys(), "mantido": rc.stats["alias"].values()})

30 -> 15 compradores


,descartado,mantido
0,C002,C001
1,C004,C003
2,C006,C005
3,C008,C007
4,C010,C009
5,C014,C013
6,C016,C015
7,C018,C017
8,C020,C019
9,C022,C021


In [4]:
rc.quarantine[["linha_origem", "motivo", "id_sobrevivente", "id_comprador", "razao_social"]]

,linha_origem,motivo,id_sobrevivente,id_comprador,razao_social
0,12,registro_vazio,NaN,C011,
1,27,registro_vazio,NaN,C026,N/A
2,3,duplicata_cnpj,C001,C002,Auto Center Pinheiros
3,5,duplicata_cnpj,C003,C004,Vidracaria do Norte
4,7,duplicata_cnpj,C005,C006,AutoPeças Sul Distribuidora
5,9,duplicata_cnpj,C007,C008,Garagem Central Rio Ltda
6,11,duplicata_cnpj,C009,C010,Vidros Express BH ME
7,15,duplicata_cnpj,C013,C014,Distribuidora PR Norte
8,17,duplicata_cnpj,C015,C016,Cristalux Reseller SC Ltda
9,19,duplicata_cnpj,C017,C018,Centro Automotivo GO


Casos especiais:

- `C011` e `C026` sao registros vazios (CNPJ zerado e `N/A`); nao sao duplicatas, vao para quarentena.
- Quase todos os CNPJs falham na validacao dos digitos verificadores (parecem fficticios). Isso nao atrapalha a deduplicacao, que so compara se dois cadastros tem o mesmo numero, mas fica registrado com o flag `cnpj_invalido`. O caso extremo e `C012` (Oficina do Ze), com `99.999.999/9999-99`; ele e mantido porque tem vendas.
- `C027`/`C028` (Parceiro ABC) tem CNPJ proprio, diferente de todos os outros, entao nao e duplicata de `C001` apesar de comecar igual.

In [5]:
rc.clean[rc.clean["flags"].map(len) > 0][["id_comprador", "razao_social", "cnpj", "flags"]]

,id_comprador,razao_social,cnpj,flags
0,C001,Auto Center Pinheiros Ltda,12345678000190,[cnpj_invalido]
1,C003,Vidraçaria do Norte ME,98765432000111,[cnpj_invalido]
2,C005,Distribuidora AutoPeças Sul,11222333000144,[cnpj_invalido]
3,C007,Garagem Central Rio,55666777000122,[cnpj_invalido]
4,C009,Vidros Express BH,33444555000166,[cnpj_invalido]
5,C012,Oficina do Zé,99999999999999,"[cnpj_invalido, data_cadastro_invalida]"
6,C013,Distribuidora Paraná Norte,44555666000177,[cnpj_invalido]
7,C015,Cristalux Reseller SC,77888999000133,[cnpj_invalido]
8,C017,Centro Automotivo Goiás,22333444000155,[cnpj_invalido]
9,C019,Reparos Rápidos SP,66777888000199,[cnpj_invalido]


## Efeito nas vendas

In [6]:
v = raws["vendas"]
alias = rc.stats["alias"]
usam_alias = v["id_comprador"].str.strip().str.upper().isin(alias.keys())
print(int(usam_alias.sum()), "linhas de venda apontam para um id de comprador descartado e foram reatribuidas ao mantido")

73 linhas de venda apontam para um id de comprador descartado e foram reatribuidas ao mantido


## Vendedores

**Chave de duplicidade:** mesmo nome normalizado (sem acento e caixa) com e-mail ou telefone iguais.

Os vendedores vizinhos no arquivo parecem pares (mesma regiao, mesmo supervisor, metas parecidas:
`V003 Carla Mendonca` e `V004 Daniel Costa`), mas sao pessoas diferentes. Nome, e-mail e telefone diferem. Nao sao duplicatas.

In [7]:
rv = res["vendedores"]
print(rv.stats["linhas_brutas"], "->", rv.stats["linhas_limpas"], "vendedores")
rv.quarantine[["linha_origem", "motivo", "id_sobrevivente", "id_vendedor", "nome"]]

20 -> 18 vendedores


,linha_origem,motivo,id_sobrevivente,id_vendedor,nome
0,20,registro_vazio,NaN,V018,
1,3,duplicata_exata,V001,V001,Ana Paula Ferreira


- `V001` aparece duas vezes identica: mantem uma.
- `V018` tem nome vazio, so um e-mail generico (`sem.nome@...`): quarentena como registro vazio.
- Os demais problemas dos vendedores sao de formato e ficam sinalizados, nao descartados:

In [8]:
rv.clean[rv.clean["flags"].map(len) > 0][["id_vendedor", "nome", "email", "data_admissao", "flags"]]

,id_vendedor,nome,email,data_admissao,flags
1,V002,Bruno Alves,bruno.alves@cristalu.com.br,2020-06-01,[email_dominio_incomum]
2,V003,Carla Mendonça,carla.mendonca@cristalux.com,2018-08-01,[email_dominio_incomum]
7,V008,Henrique Souza,henrique.souza@cristalux.com,2020-05-05,[email_dominio_incomum]
13,V014,Nicolas Andrade,NaN,2023-03-03,[email_ausente]
17,V019,Rafael Cunha,rafael.cunha@cristalux.com.br,None,[admissao_invalida]


`V019` tem admissao em 2030 (futuro): o campo fica nulo e o registro e mantido com o flag `admissao_invalida`.
Dominios de e-mail diferentes do corporativo (`cristalu.com.br`, `cristalux.com`) sao sinalizados e nao corrigidos, porque nao ha como saber o certo.